# Visualizations



In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, "scripts")

import numpy as np

from config import CONFIG
from support.common import corpus, read_csv
from support.tables import (
    read_frames, read_retrieval_results,
    read_synthesis_results, read_synthesis_annotations,
)
from support.metrics import calculate_events, summarize_events, analyze_characteristics

# Saved frame scores and selections are enough to reproduce the analysis.
articles = corpus("data_public/corpus_public.csv")
frame_definitions = read_frames("data_public/frames_public.csv")
query_labels = read_csv("data_public/queries_public.csv")
article_scores = read_csv(
    "data_public/article_annotations_public.csv", usecols=["article_id", "frame_id", "score"],
)
answer_scores = read_synthesis_annotations(
    ["data_public/annotated_synthesis_results_public.csv"],
)
retrieval = read_retrieval_results(
    ["data_public/retrieval_results_public.csv.gz"], query_labels,
)
answers = read_synthesis_results(
    ["data_public/synthesis_results_public.csv.gz"], query_labels,
)
events = sorted(frame_definitions.event_name.unique())

# Use the same 5,000 whole-event resamples for paired method/reference comparisons.
bootstrap = np.random.default_rng(CONFIG["bootstrap_seed"]).integers(
    0, len(events), (CONFIG["bootstrap_draws"], len(events)),
)
estimates = calculate_events(
    articles, frame_definitions, article_scores, answer_scores,
    retrieval, answers, query_labels,
)
summary = summarize_events(estimates, bootstrap)
characteristics, frame_shifts = analyze_characteristics(estimates, frame_definitions, bootstrap)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
models = CONFIG["models"]
names = ["GPT", "Claude", "Gemini"]
colors = ["#4C78A8", "#D18743", "#5D9275"]
figures = Path("figures")
figures.mkdir(exist_ok=True)


def bar(ax, x, row, color, width=.32, hatch=""):
    """Draw an estimate, its bootstrap interval, and a star if the interval excludes zero."""
    ax.bar(x, row["mean"], width, color=color, hatch=hatch, edgecolor="black", linewidth=.4)
    ax.errorbar(
        x, row["mean"], yerr=[[row["mean"] - row.ci_low], [row.ci_high - row["mean"]]],
        fmt="none", color="black", capsize=2, linewidth=.7,
    )
    if row.ci_low > 0 or row.ci_high < 0:
        # Put the star outside the interval, away from zero.
        positive = row.ci_low > 0
        ax.annotate(
            "*", (x, row.ci_high if positive else row.ci_low),
            xytext=(0, 3 if positive else -3), textcoords="offset points",
            ha="center", va="bottom" if positive else "top",
        )


def finish(fig, name, rect=None):
    """Display the figure and save PDF and PNG copies."""
    fig.tight_layout(rect=rect)
    fig.savefig(figures / (name + ".pdf"), bbox_inches="tight")
    fig.savefig(figures / (name + ".png"), dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)


# Retrieval


In [ ]:
# Exposure bars are paired method-minus-Random differences; hatching denotes centrality.
methods = ["bm25_headline", "bm25_full", "dense_headline", "dense_first_chunk"] + models
labels = ["BM25\nheadlines", "BM25\nfull body", "Dense\nheadlines", "Dense\nfirst chunk"] + names
palette = ["#A392B1", "#786385", "#B1A08D", "#88745F"] + colors
fig, axes = plt.subplots(2, 1, figsize=(8.4, 5.5))
for method_index, (method, color) in enumerate(zip(methods, palette)):
    for metric, offset, hatch in [("any_shift_pp", -.17, ""), ("central_shift_pp", .17, "///")]:
        row = summary.query(
            "stage == 'retrieval_random' and "
            "model == @method and "
            "metric == @metric"
        ).iloc[0]
        bar(axes[0], method_index + offset, row, color, hatch=hatch)
    row = summary.query(
        "stage == 'retrieval_random' and "
        "model == @method and "
        "metric == 'mention_tvd'"
    ).iloc[0]
    bar(axes[1], method_index, row, color, width=.6)
for ax in axes:
    ax.set_xticks(range(7), labels)
    ax.axhline(0, color="black", linewidth=.6)
axes[0].set_title("A  Change in frame salience relative to Random", loc="left")
axes[0].set_ylabel("Method − Random (pp)")
axes[1].set_title("B  Relative frame distribution TVD above Random", loc="left")
axes[1].set_ylabel("Method TVD − Random TVD")
axes[0].legend(handles=[Patch(facecolor="white", edgecolor="black", label="Mentioned"),
                       Patch(facecolor="white", edgecolor="black", hatch="///", label="Central")], frameon=False)
finish(fig, "retrieval")


In [ ]:
# Appendix quantities keep the corpus reference instead of subtracting Random.
fig, axes = plt.subplots(2, 1, figsize=(8.4, 5))
for ax, metric, title in zip(axes, ["variance_change", "any_shift_pp"],
                           ["A  Within-article variance change", "B  Frame coverage change (pp)"]):
    for method_index, (method, color) in enumerate(zip(methods, palette)):
        row = summary.query(
            "stage == 'retrieval_corpus' and "
            "model == @method and "
            "metric == @metric"
        ).iloc[0]
        bar(ax, method_index, row, color, width=.6)
    ax.set_xticks(range(7), labels)
    ax.axhline(0, color="black", linewidth=.6)
    ax.set_title(title, loc="left")
finish(fig, "retrieval_variance_coverage")


# Synthesis


In [ ]:
# A curve bin averages the events with observations at that input-article count.
def curve(ax, axis, outcome, title, paper_wording=False):
    """Plot answer rates against the number of input articles with the frame."""
    for model, color, label in zip(models, colors, names):
        rows = summary[(summary.stage == "curve") & (summary.model == model) &
                       summary.metric.str.startswith(axis + ":" + outcome + ":")].copy()
        rows["level"] = rows.metric.str.split(":").str[-1].astype(int)
        rows = rows.sort_values("level")
        ax.plot(rows.level, rows["mean"], "o-", color=color, markersize=3, label=label)
        ax.fill_between(rows.level, rows.ci_low, rows.ci_high, color=color, alpha=.15)
    # Equal rates means the answer probability equals the input count divided by ten.
    if axis == outcome:
        ax.plot([0, 10], [0, 1], "--", color="gray", linewidth=.8,
                label="Equal rates" if paper_wording else None)
    ax.set(xlim=(0, 10), ylim=(0, 1), xticks=range(11),
           xlabel="Input articles with frame " + axis + " (of 10)",
           ylabel="Output mention rate" if outcome == "mentioned" else "Output central rate")
    if paper_wording:
        ax.set_xlabel("Input articles mentioning frame (of 10)" if axis == "mentioned"
                      else "Input articles with frame central (of 10)")
        ax.set_ylabel("Response mention rate" if outcome == "mentioned" else "Response central-frame rate")
    ax.set_title(title, loc="left")

fig, axes = plt.subplots(1, 3, figsize=(12, 2.7))
for model_index, (model, color) in enumerate(zip(models, colors)):
    for metric, offset, hatch in [("any_shift_pp", -.17, ""), ("central_shift_pp", .17, "///")]:
        row = summary.query(
            "stage == 'synthesis' and "
            "model == @model and "
            "metric == @metric"
        ).iloc[0]
        bar(axes[0], model_index + offset, row, color, hatch=hatch)
    # This contrast subtracts the exact, nonempty Bernoulli reference from observed TVD.
    row = summary.query("stage == 'synthesis' and model == @model and metric == 'excess'").iloc[0]
    bar(axes[2], model_index, row, color, width=.6)
axes[0].set_title("A  Change in frame salience", loc="left")
axes[0].set_ylabel("Change in frame rate (pp)")
axes[0].set_xticks(range(3), names)
axes[0].axhline(0, color="black", linewidth=.6)
axes[0].legend(handles=[Patch(facecolor="white", edgecolor="black", label="Mentioned"),
                       Patch(facecolor="white", edgecolor="black", hatch="///", label="Central")], frameon=False)
curve(axes[1], "mentioned", "mentioned", "B  Mentioned → mentioned")
axes[1].legend(frameon=False, fontsize=7)
axes[2].set_title("C  Relative frame distribution TVD", loc="left")
axes[2].set_ylabel("Observed − reference TVD")
axes[2].set_xticks(range(3), names)
axes[2].axhline(0, color="black", linewidth=.6)
finish(fig, "synthesis")


In [ ]:
# Cross-threshold panels have no equal-rate guide because their two indicators differ.
fig, axes = plt.subplots(2, 2, figsize=(8, 5.8))
for ax, (input_level, output_level, title) in zip(axes.flat, [
    ("mentioned", "mentioned", "A  Mentioned → mentioned"),
    ("mentioned", "central", "B  Mentioned → central"),
    ("central", "mentioned", "C  Central → mentioned"),
    ("central", "central", "D  Central → central")]):
    curve(ax, input_level, output_level, title, paper_wording=True)
axes[0, 0].legend(frameon=False)
finish(fig, "synthesis_transfer_curves")


# Frame characteristics


In [ ]:
# Each point is one event-specific frame; the fitted line is descriptive.
paper_names = ["GPT-5.4", "Claude Opus 4.6", "Gemini 3.1 Pro"]
frames = frame_shifts.merge(frame_definitions[["event_name", "frame_id", "slant"]],
                      left_on=["event", "frame"], right_on=["event_name", "frame_id"])
fig, axes = plt.subplots(2, 3, figsize=(9, 5.5))
fig.suptitle("Frame Score Shift vs. Political Slant", fontsize=13)
fig.supxlabel("Frame political slant")
fig.supylabel("Frame score shift")
for stage_index, stage in enumerate(["retrieval", "synthesis"]):
    fig.text(.055, .72 if stage_index == 0 else .31, f"{stage_index + 1}. {stage.title()}",
             rotation=90, ha="center", va="center", fontweight="bold")
    for model_index, (model, color, name) in enumerate(zip(models, colors, paper_names)):
        ax = axes[stage_index, model_index]
        frame_points = frames.query("stage == @stage and model == @model")
        ax.scatter(frame_points.slant, frame_points["shift"], color=color, s=8, alpha=.6)
        slope, intercept = np.polyfit(frame_points.slant, frame_points["shift"], 1)
        ax.plot([-1, 1], np.array([-1, 1]) * slope + intercept, color=color, linewidth=.8)
        row = characteristics.query(
            "stage == @stage and "
            "model == @model and "
            "characteristic == 'political_slant_score'"
        ).iloc[0]
        significant = row.ci_low > 0 or row.ci_high < 0
        star = " *" if significant else ""
        ax.text(.97, .03, f"r = {row['mean']:.2f}{star}\n[{row.ci_low:.2f}, {row.ci_high:.2f}]",
                ha="right", va="bottom", transform=ax.transAxes, fontsize=8,
                fontweight="bold" if significant else "normal")
        ax.axhline(0, color="gray", linestyle="--", linewidth=.6)
        ax.set_title(name if stage_index == 0 else "")
        ax.set_xlim(-1, 1)
        ax.set_xticks([-1, 0, 1], ["-1\nleft", "0", "1\nright"] if stage_index == 1 else [])
finish(fig, "political_slant", rect=(.07, .06, 1, .94))


In [ ]:
# Each bar compares frames with a label against all frames without that label.

families = ["entman_functions", "sv_frame", "card_categories"]
levels = [["define problems", "diagnose causes", "make moral judgments", "suggest remedies"],
          ["conflict", "human interest", "economic", "morality", "responsibility"],
          ["political", "legal", "economic", "human-wellbeing", "values", "cultural", "public-opinion"]]
family_titles = ["Entman Frame Function", "Semetko & Valkenburg Frame Type", "Condensed Card Frame Category"]
wrapped_labels = {"make moral judgments": "make moral\njudgments",
                  "human-wellbeing": "human-\nwellbeing", "public-opinion": "public-\nopinion"}


def plot_characteristics(columns, filename):
    """Compare shifts for frames with each label against frames without it."""
    widths = [4, 7, 5] if columns == 3 else [4, 5]
    fig, axes = plt.subplots(2, columns, figsize=(4.2 * columns, 6), squeeze=False,
                             gridspec_kw={"width_ratios": widths})
    title = "Which frames do LLM retrieval and synthesis raise more?"
    if columns == 3:
        title += "\n(* event-bootstrap 95% CI excludes 0)"
    fig.suptitle(title, fontsize=12)
    if columns == 2:
        fig.supylabel("Δ frame score shift")
    for stage_index, stage in enumerate(["retrieval", "synthesis"]):
        fig.text(.055 if columns == 2 else .02, .73 if stage_index == 0 else .35,
                 f"{stage_index + 1}. {stage.title()}", rotation=90, ha="center", va="center", fontweight="bold")
        selected = [0, 2, 1] if columns == 3 else [0, 1]
        for panel_index, index in enumerate(selected):
            family, labels = families[index], levels[index]
            ax = axes[stage_index, panel_index]
            for model_index, (model, color) in enumerate(zip(models, colors)):
                for label_index, label in enumerate(labels):
                    row = characteristics.query(
                        "stage == @stage and "
                        "model == @model and "
                        "family == @family and "
                        "characteristic == @label"
                    ).iloc[0]
                    bar(ax, label_index + (model_index - 1) * .23, row, color, width=.21)
            ax.axhline(0, color="black", linewidth=.6)
            ticklabels = []
            for label in labels:
                count = characteristics.query("family == @family and characteristic == @label").iloc[0].n_frames
                wording = wrapped_labels.get(label, label.replace(" ", "\n"))
                ticklabels.append(f"{wording}\n(n = {int(count)})")
            ax.set_xticks(range(len(labels)), ticklabels if stage_index == 1 else [], fontsize=7)
            ax.set_title(family_titles[index] if stage_index == 0 else "", fontsize=10)
            if columns == 3 and panel_index == 0:
                ax.set_ylabel("difference in frame score shift\n(frames with the value −\nframes without)", fontsize=8)
    handles = [Patch(facecolor=c, label=n) for c, n in zip(colors, paper_names)]
    handles.append(Line2D([], [], color="black", label="95% bootstrap CI"))
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=8)
    finish(fig, filename, rect=(.07 if columns == 2 else .04, .07, 1, .90))


plot_characteristics(2, "frame_characteristics")
plot_characteristics(3, "frame_characteristics_all")
